In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy


train = pd.read_excel("BNT-Energy2f.xlsx")
X=train.iloc[:,:-1]
Y=train.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


scalerX = StandardScaler().fit(X_train)
scalery = StandardScaler().fit(y_train.values.reshape(-1,1))
X_train1 = scalerX.transform(X_train)
y_train1 = scalery.transform(y_train.values.reshape(-1,1))
X_test1 = scalerX.transform(X_test)
y_test1 = scalery.transform(y_test.values.reshape(-1,1))
X_train2=copy.deepcopy(X_train1)
y_train2=copy.deepcopy(y_train1)



from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from xgboost import XGBRegressor
# from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

# Set a random state for reproducibility
RANDOM_STATE = 42

xgb_model=XGBRegressor(random_state=RANDOM_STATE)

search_space={'n_estimators':[100,200,300,400,500,600,800],'max_depth':[3,4,6,7,9],'gamma0':[0.01],'learning_rate':[0.001,0.01,0.05,0.1,0.5,1]}


from sklearn.model_selection import GridSearchCV

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
GS=GridSearchCV(estimator=xgb_model,param_grid=search_space,scoring=["r2","neg_root_mean_squared_error"],refit="neg_root_mean_squared_error",cv=kf,verbose=4)
GS.fit(X_train1,y_train1)

print(GS.best_params_) # ND gamma0=0.01, learning_rate=0.01, max_depth=9, n_estimators=800;
print(GS.best_score_)

# Define best_xgb_model here to make it accessible for subsequent cells
best_xgb_model = GS.best_estimator_



### SHAP Feature Importance

SHAP (SHapley Additive exPlanations) is a game theoretic approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations by using Shapley values from game theory.

In [1]:
import shap

exp = shap.TreeExplainer(best_xgb_model)
shap_values = exp.shap_values(X_train1)

g_s_important = np.abs(shap_values).mean(axis=0)
shap_imp_file = pd.DataFrame({'Feature': X.columns,'Global SHAP Importance': g_s_important}).sort_values(by='Global SHAP Importance', ascending=False)

print("Global Feature Importance (Mean Absolute SHAP Values):")
display(shap_imp_file)

NameError: name 'best_xgb_model' is not defined

In [2]:
feature_imps = best_xgb_model.feature_importances_
feature = X.columns

feature_imp = pd.DataFrame({'Feature': feature, 'Importance': feature_imps})
feature_imp = feature_imp.sort_values(by='Importance', ascending=False)

print("Relative Feature Importance:")
display(feature_imp)


NameError: name 'best_xgb_model' is not defined